# Modelo de Regressão para prever os custos médicos

Chegou a hora de compilar todos os aprendizados obtidos nas aulas de Machine Learning, em um único lugar, para fazer a entrega do Desafio da disciplina. Utilizando a base de dados "insurance.csv", você tem o desafio de criar um modelo preditivo de regressão para prever o valor dos custos médicos individuais cobrados pelo seguro de saúde.

## Sobre a base de dados
Essa base de dados contém 1.338 linhas com informações sobre a idade da pessoa, gênero, índice de massa corporal (IMC), número de filhos, flag de verificação se a pessoa é fumante, região residencial do benefício e o valor do custo médico.

Dicionário dos dados
* Idade: idade do beneficiário principal.
* Gênero: gênero do contratante de seguros.
*  IMC: Índice de massa corporal, fornecendo uma compreensão do corpo, pesos relativamente altos ou baixos em relação à altura.
* Filhos: número de filhos cobertos por seguro saúde / Número de dependentes.
* Fumante: se a pessoa fuma (sim ou não).
* Região: a área residencial do beneficiário nos EUA, nordeste, sudeste, sudoeste ou noroeste.
* Encargos: custos médicos individuais cobrados pelo seguro de saúde.

## Objetivo
Criar um modelo preditivo e comprovar sua eficácia com métricas estatísticas.

In [ ]:
# Importando as bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr
import statsmodels.api as sm
from statsmodels.formula.api import ols
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR


In [ ]:
# Carregando os dados
df = pd.read_csv('dataset/insurance.csv')
df.head()

In [ ]:
df.info()

In [ ]:
# Verificando valores nulos
df.isnull().sum()

Não há valores ausentes

In [ ]:
# Verificando valores duplicados
print(f'Quantidade de linhas duplicadas: {df.duplicated().sum()}')

# Verificando qual valor está duplicado
print('\nVerificando qual linha tá duplicada:')
display(df[df.duplicated()])

# Filtrando pela idade de 19 anos, masculino e imc
print('\nVisualizando o valor duplicado:')
display(df[(df['age'] == 19) & (df['sex'] == 'male') & (df['bmi'] == 30.59)])

O nosso dataset possui um valor duplicado, entretanto esse valor pode ser de pessoas diferentes, pois duas pessoas podem ter as mesmas características, por isso, não será deletado.

## Análise descritiva e exploratória

In [ ]:
df.describe()

### Idade

In [ ]:
# Histograma de idades
plt.figure(figsize=(10, 6))
sns.histplot(df['age'], bins=60)
plt.title('Distribuição de Idades')
plt.xlabel('Idade')
plt.ylabel('Frequência')
plt.show()

In [ ]:
sns.boxplot(x=df['age'])
plt.title('Boxplot da Idade')
plt.xlabel('Idade')
plt.show()

### IMC

In [ ]:
sns.histplot(df['bmi'], bins=30, color='green')
plt.title('Distribuição do IMC')
plt.xlabel('IMC')
plt.ylabel('Frequência')
plt.show()

In [ ]:
# Verificando se IMC possui distribuição normal
stat_sw, p_sw = stats.shapiro(df['bmi'])
print(f'Estatistica de Shapiro-Wilk: {stat_sw}, p-valor:{p_sw}')

In [ ]:
sns.boxplot(x=df['bmi'], color='green')
plt.title('Boxplot do IMC')
plt.xlabel('IMC')
plt.show()  

### Número de Filhos

In [ ]:
sns.histplot(df['children'], bins=5, color='orange')
plt.title('Distribuição do Número de Filhos')
plt.xlabel('Número de Filhos')
plt.ylabel('Frequência')
plt.show()

In [ ]:
sns.boxplot(x=df['children'], color='orange')
plt.title('Boxplot de Filhos')
plt.xlabel('Número de Filhos')
plt.show()

### Custos médicos

In [ ]:
sns.histplot(df['charges'], bins=50, color='red')
plt.title('Distribuição dos Custos Médicos')
plt.xlabel('Custos Médicos')
plt.ylabel('Frequência')
plt.show()

In [ ]:
sns.boxplot(x=df['charges'], color='red')
plt.title('Boxplot dos Custos Médicos')
plt.xlabel('Custos Médicos')
plt.show()

## Teste de Hipótese

### Há diferença entre os custos médios entre os sexos?

* H0: Não há diferença significativa de custos entre os gêneros
* H1: Há diferença significativa de custos entre os gêneros

In [ ]:
# Teste t de Student para verificar se há diferença
male = df[df['sex']=='male']['charges']
female = df[df['sex']=='female']['charges']

t_stat, p_value = stats.ttest_ind(male, female, equal_var=False)
print(f'Estatistica t: {t_stat}, p-valor: {p_value}')

p-valor < 0,05, então rejeitamos H0. 

Então há diferença entre as médias dos custos entre os gêneros. Também podemos verificar com o gráfico a seguir:

In [ ]:
# Sexo e custos médicos
sns.boxplot(x='sex', y='charges', data=df)
plt.title('Sexo vs Custos Médicos')
plt.xlabel('Sexo')
plt.ylabel('Custos Médicos')
plt.show()

No gráfico, temos que o sexo masculino tem maiores custos médicos comparado com as mulheres.

### Há diferença entre os custos médios de fumantes e não-fumantes?

* H0: Não há diferença significativa nos custos médios entre fumantes e não-fumantes.
* H1: Há diferença significativa nos custos médios entre fumantes e não-fumantes.

In [ ]:
smoker = df[df['smoker']=='yes']['charges']
no_smoker = df[df['smoker']=='no']['charges']

t_stat_smoker, p_value_smoker = stats.ttest_ind(smoker, no_smoker, equal_var=False)
print(f'Estatistica t: {t_stat_smoker}, p-valor: {p_value_smoker}')

p-valor < 0,05, rejeitamos H0. 

Então há diferença entre as médias dos custos entre os fumantes e não fumantes. 

In [ ]:
# Fumante e custos médicos
sns.boxplot(x='smoker', y='charges', data=df)
plt.title('Fumante vs Custos Médicos')
plt.xlabel('Fumante')
plt.ylabel('Custos Médicos')
plt.show()

No gráfico podemos observar que os custos médios para os fumantes é bem maior do que para os não fumantes.

### Os custos médios são iguais em todas as Regiões?

* H0: todos os custos médios por região são iguais
* H1: Pelo menos uma média de região é diferente das demais

In [ ]:
# Usando statsmodels:
formula = 'charges ~ C(region)'
model = ols(formula, data=df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
anova_table

In [ ]:
# Usando scipy.stats:
northwest = df[df['region']=='northwest']['charges']
northeast = df[df['region']=='northeast']['charges']
southeast = df[df['region']=='southeast']['charges']
southwest = df[df['region']=='southwest']['charges']
f_statistic, p_value = stats.f_oneway(northwest, northeast, southeast, southwest)
print(f'\nEstatística F: {f_statistic:.4}, p-valor: {p_value:.4}')

p-value < 0.05. Isso significa que rejeitamos a Hipótese Nula. 

Há uma diferença estatisticamente significativa nas médias dos custos entre pelo menos um par de regiões.

In [ ]:
# Região e custos médicos
sns.boxplot(x='region', y='charges', data=df)
plt.title('Região vs Custos Médicos')
plt.xlabel('Região')
plt.ylabel('Custos Médicos')
plt.show()

A região Sudeste (southeast) apresenta a média de custos mais alta, o que provavelmente é o principal fator que impulsiona a diferença estatisticamente significativa.

### Todos os custos médios por número de dependentes são iguais?

* H0: Todos os custos médios de seguro por região são iguais.
* H1: Pelo menos uma média de dependentes é diferente das demais.

In [ ]:
# Usando statsmodels:
formula = 'charges ~ C(children)'
model = ols(formula, data=df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
anova_table

In [ ]:
# Filhos e custos médicos
sns.boxplot(x='children', y='charges', data=df)
plt.title('Filhos vs Custos Médicos')
plt.xlabel('Filhos')
plt.ylabel('Custos Médicos')
plt.show()

### Há correlação entre Idade e Custos?

* H0: Não há correlação linear entre a idade e os custos.
* H1: Há correlação linear entre a idade e os custos.

In [ ]:
r_age, p_value_age = pearsonr(df['age'], df['charges'])
print(f'Correlação de Pearson: {r_age:.4}, p_valor: {p_value_age:.4}')

Correlação positiva fraca/moderada e altamente significativa. Conforme a idade aumenta, os custos tendem a aumentar.

In [ ]:
# Os custos médicos são influenciados pela idade?
sns.scatterplot(data=df, x='age', y='charges')
plt.title('Idade vs Custos Médicos')
plt.xlabel('Idade')
plt.ylabel('Custos Médicos')
plt.show()

### Há correlação entre IMC e Custos?

* H0: Não há correlação linear entre imc e os custos.
* H1: Há correlação linear entre imc e os custos.

In [ ]:
# Correlação de Pearson
r_imc, p_value_imc = pearsonr(df['bmi'], df['charges'])
print(f'Correlação de Pearson: {r_imc:.4}, p_valor: {p_value_imc:.4}')

Correlação positiva fraca e altamente significativa. Conforme o IMC aumenta, os custos tendem a aumentar.

In [ ]:
# Qual o impacto do IMC nos custos médicos?
sns.scatterplot(data=df, x='bmi', y='charges')
plt.title('IMC vs Custos Médicos')
plt.xlabel('IMC')
plt.ylabel('Custos Médicos')
plt.show()

### Matriz de Correlação

In [ ]:
# Transformando sexo em variável binário
df_encoded = df.copy()
df_encoded['sex'] = df_encoded['sex'].map({'male': 0, 'female': 1})

# Transformando fumante em variável binário
df_encoded['smoker'] = df_encoded['smoker'].map({'no': 0, 'yes': 1})

# Transformando região em variáveis dummy
df_encoded = pd.get_dummies(df_encoded, columns=['region'])    

# Visualizando o dataframe codificado
df_encoded.head()

In [ ]:
# Correlação entre as variáveis numéricas
plt.figure(figsize=(10,6))  
sns.heatmap(df_encoded.corr(), annot=True, cmap='coolwarm')
plt.title('Matriz de Correlação')
plt.show()

In [ ]:
# Separando features e target
X = df_encoded.drop('charges', axis=1)
y_original = df_encoded['charges']
y_log = np.log(y_original)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42)   
print(f'Tamanho do conjunto de treino: {X_train.shape[0]}')
print(f'Tamanho do conjunto de teste: {X_test.shape[0]}')

In [ ]:
# Padronizando e normalizando os dados

# Inicializar o scaler
scaler_X = StandardScaler() # Para as features
scaler_y = StandardScaler() # Para a variável target

# Colunas numéricas que precisam de padronização
numeric_cols = ['age', 'bmi', 'children'] 

# Padronizar as FEATURES (X)
X_train[numeric_cols] = scaler_X.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler_X.transform(X_test[numeric_cols])

# Padronizar a VARIÁVEL TARGET (Y)
# Deve ser feita a transformação para que os modelos treinem na mesma escala.
# É importante remodelar (reshape) para (n_samples, 1) antes de usar o scaler
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1))
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1)) 

# O y_train/y_test agora são arrays NumPy na nova escala. 
# Para manter a consistência, remova o reshaping no final, se necessário.
y_train_scaled = y_train_scaled.flatten()
y_test_scaled = y_test_scaled.flatten()

In [ ]:
# --- 5. Define Inverse Transformation Function ---
# This function reverses both the scaling and the log transformation
def inverse_transform_y(y_scaled_pred, scaler, is_log=True):
    # 1. Inverse scaling
    y_log_pred = scaler.inverse_transform(y_scaled_pred.reshape(-1, 1)).flatten()
    if is_log:
        # 2. Inverse log transformation (Exponentiation)
        return np.exp(y_log_pred)
    return y_log_pred

# Get the original (non-log) y_test values corresponding to the test split
y_test_original = y_original.loc[y_test.index]

# --- 6. Train and Evaluate All Models ---
results = []

# Modelos

## Usando o Modelo de Regressão Linear

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train_scaled)

# Fazendo previsões
y_pred = model.predict(X_test)
y_pred_original = inverse_transform_y(y_pred, scaler_y, is_log=True)

# Avaliando o modelo
mae = mean_absolute_error(y_test_scaled, y_pred_original)
mse = mean_squared_error(y_test_scaled, y_pred_original)
r2 = r2_score(y_test_scaled, y_pred_original)

## Usando o Modelo de Árvore de Decisão

In [ ]:
model_dt = DecisionTreeRegressor(random_state=42)
model_dt.fit(X_train, y_train_scaled)

# Fazendo previsões
y_pred_dt = model_dt.predict(X_test)

# Avaliando o modelo de árvore de decisão
mae_dt = mean_absolute_error(y_test_scaled, y_pred_dt)
mse_dt = mean_squared_error(y_test_scaled, y_pred_dt)  
r2_dt = r2_score(y_test_scaled, y_pred_dt)

## Usando o RandomForestRegressor

In [ ]:
model_rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
model_rf.fit(X_train, y_train_scaled)

# Fazendo previsões
y_pred_rf = model_rf.predict(X_test)

# Avaliando o modelo Random Forest
mae_rf = mean_absolute_error(y_test_scaled, y_pred_rf)
mse_rf = mean_squared_error(y_test_scaled, y_pred_rf)
r2_rf = r2_score(y_test_scaled, y_pred_rf)


## Usando o modelo svm

In [ ]:
model_svr = SVR(kernel='linear')
model_svr.fit(X_train, y_train_scaled)

# Fazendo previsões
y_pred_svr = model_svr.predict(X_test)

# Avaliando o modelo SVM
mae_svr = mean_absolute_error(y_test_scaled, y_pred_svr)
mse_svr = mean_squared_error(y_test_scaled, y_pred_svr)
r2_svr = r2_score(y_test_scaled, y_pred_svr)

# Tabela de Comparação entre os modelos

In [ ]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Decision Tree', 'Random Forest', 'SVM'],
    'MAE': [mae, mae_dt, mae_rf, mae_svr],
    'MSE': [mse, mse_dt, mse_rf, mse_svr],
    'R^2': [r2, r2_dt, r2_rf, r2_svr]
})

results